# Visualising decision trees (dtreeviz and the alternatives used here)
dtreeviz needs matplotlib and Graphviz, so this Notebook does not import it. The dtreeviz calls are shown in Markdown;
every result is rebuilt with scikit-learn, Plotly and Graphviz (`images/figs.py`, `images/trees.py`).

```python
# pip install dtreeviz        (also needs the Graphviz program `dot`)
import dtreeviz
viz = dtreeviz.model(clf, X_train=X, y_train=y,
                     feature_names=iris.feature_names,
                     target_name="flower", class_names=list(iris.target_names))
viz.view()                                   # the tree with histograms and pies
viz.view(scale=2)                            # bigger
viz.view(orientation="LR")                   # left to right
viz.view(x=x)                                # highlight one row's prediction path
viz.view(x=x, show_just_path=True)           # draw only that path
viz.view(show_node_labels=True)              # node numbers
viz.view(fancy=False)                        # no histograms, just boxes and pies
print(viz.explain_prediction_path(x))        # the path in plain English
viz.instance_feature_importance(x)           # importance along one row's path
viz.rtree_feature_space(features=["WGT"])    # regression tree on one input
viz.rtree_feature_space3D(features=["WGT", "ENG"])   # on two inputs, in 3D
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

## The iris tree, depth 2

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
clf = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
# show_weights=True prints how many rows of each class reach every leaf
print(export_text(clf, feature_names=iris.feature_names, show_weights=True))

## The fully grown tree: node numbers and one flower's prediction path

In [ ]:
full = DecisionTreeClassifier(random_state=0).fit(X, y)
print("nodes:", full.tree_.node_count, "| depth:", full.get_depth(), "| leaves:", full.get_n_leaves())

x = X[70]                                     # one flower (the same one as in Figure 3 of the Note)
print("flower:", dict(zip(iris.feature_names, x)), "->", iris.target_names[full.predict([x])[0]])
path = full.decision_path([x]).indices        # node numbers visited, root first
print("path:", [int(n) for n in path])

In [ ]:
def explain_path(tree, x, names):
    # Plain-English path, like dtreeviz's explain_prediction_path: one range per feature used
    t = tree.tree_
    low, high = {}, {}
    for node in tree.decision_path([x]).indices:
        if t.children_left[node] == -1:
            break                              # reached the leaf
        f, thr = t.feature[node], t.threshold[node]
        if x[f] <= thr:
            high[f] = min(high.get(f, np.inf), thr)
        else:
            low[f] = max(low.get(f, -np.inf), thr)
    for f in sorted(set(low) | set(high)):
        left = f"{low[f]:.2f} < " if f in low else ""
        right = f" <= {high[f]:.2f}" if f in high else ""
        print(f"{left}{names[f]}{right}")

explain_path(full, x, iris.feature_names)

In [ ]:
# Feature importance of the fully grown tree
pd.Series(full.feature_importances_, index=iris.feature_names).sort_values(ascending=False).round(3)

## A regression split on the Boston data (all 506 districts, depth 1)

In [ ]:
boston = pd.read_csv("data/boston.csv")
reg = DecisionTreeRegressor(max_depth=1, random_state=0).fit(boston.drop(columns="MEDV"), boston["MEDV"])
t = reg.tree_
print("split:", boston.columns[t.feature[0]], "<=", round(t.threshold[0], 2))
print("left mean", round(t.value[1].item(), 2), "n =", t.n_node_samples[1],
      "| right mean", round(t.value[2].item(), 2), "n =", t.n_node_samples[2])

## Cars: one input and two inputs

In [ ]:
cars = pd.read_csv("data/cars.csv")         # the sample data shipped with dtreeviz
uni = DecisionTreeRegressor(max_depth=3, random_state=0).fit(cars[["WGT"]], cars["MPG"])
print("cuts on WGT:", sorted(float(v) for v in uni.tree_.threshold[uni.tree_.children_left != -1]))
bi = DecisionTreeRegressor(max_depth=3, random_state=0).fit(cars[["WGT", "ENG"]], cars["MPG"])
print(export_text(bi, feature_names=["WGT", "ENG"]))